### Create a baseline to benchmark models on the Oct22 screening data

In [1]:
import pathlib
import yaml

import dataset_Oct22

In [2]:
data_dir = pathlib.Path("../output")
data_csv_fn = data_dir / "ordinal_Oct22_sequences_with_dca_score.csv"
ds = dataset_Oct22.Oct22DataSet(data_csv_fn=data_csv_fn)
print(ds)


aa_length :272
num_seqs  :1326
num_train :1127
num_test  :199



In [3]:
add_dca = False
activity_only = False

In [4]:
# get train data
train_data = ds.get_train_dataset()
X_train = dataset_Oct22.create_model_inputs(train_data, add_dca=add_dca)
y_train = dataset_Oct22.create_target(train_data, activity_only=activity_only)
print(X_train.shape, y_train.shape)

(1127, 5440) (1127,)


In [5]:
from sklearn.linear_model import RidgeClassifierCV

In [6]:
model = RidgeClassifierCV(fit_intercept=True, cv=ds)
clf = model.fit(X_train, y_train)

In [7]:
# get test data
test_data = ds.get_test_dataset()
X_test = dataset_Oct22.create_model_inputs(test_data, add_dca=add_dca)
y_test = dataset_Oct22.create_target(test_data, activity_only=activity_only)

In [8]:
model.score(X_test, y_test)

0.5276381909547738

In [9]:
decision = model.decision_function(X_test)

In [10]:
from sklearn.utils.extmath import softmax

In [11]:
des = model.decision_function(X_test)

In [12]:
des.shape

(199, 4)

In [13]:
probs = softmax(des)

In [15]:
probs[:10]

array([[0.13056899, 0.67662901, 0.09187585, 0.10092615],
       [0.28069424, 0.42631259, 0.15570056, 0.13729262],
       [0.33698895, 0.27010992, 0.22978839, 0.16311274],
       [0.33523521, 0.31601085, 0.21682447, 0.13192948],
       [0.20701049, 0.50885933, 0.14286654, 0.14126364],
       [0.24053288, 0.46124554, 0.13526182, 0.16295976],
       [0.19223177, 0.52034481, 0.15874816, 0.12867526],
       [0.18710717, 0.53386812, 0.15016048, 0.12886424],
       [0.25945969, 0.47812961, 0.12744404, 0.13496666],
       [0.2369719 , 0.25849798, 0.35325053, 0.15127958]])

In [50]:
import hashlib

class ModelConfig:
  
    def __init__(self, model_name, 
                 target="multiclass", # binary or multiclass 
                 intercept=False, # whether the design matrix has an intercept or not
                 dca=False,  # whether dca is added to design matrix
                 multilibrary=False, # whether we consider separate libraries
                 encoding="one-hot", # one-hot, esm, esmft
                 uuid=None
                ):
      
        self.model_name = model_name
        self.target = target
        self.intercept = intercept
        self.dca = dca
        self.multilibrary = multilibrary
        self.encoding = encoding

        self.uuid = uuid
        if self.uuid is None:
            self.uuid = self.get_uuid()
      
    def get_uuid(self):
        """ hash a text representation of this classes dict"""
        variables_d = self.__dict__.copy()
        variables_d.pop("uuid") # remove uuid so that it is consistent if called twice
        return hashlib.blake2b(bytes(variables_d.__repr__(), "utf-8"), 
                             digest_size=4).hexdigest()
    
    def __repr__(self):
        return self.__dict__.__repr__()

In [52]:
ModelConfig("t")

{'model_name': 't', 'target': 'multiclass', 'intercept': False, 'dca': False, 'multilibrary': False, 'encoding': 'one-hot', 'uuid': 'dfa88c76'}

In [53]:
t.get_uuid()

'dfa88c76'

'dfa88c76'

In [ ]:
yaml_dict = {
  'model_name': "SKlearnRidgeClassifierCV",
  'target': "binary" if activity_only else "multiclass"
  'design_matrix' {}
  'multilibrary':False
dca = add_dca